In [1]:
!wget "https://portal.nersc.gov/cfs/m4392/G25/Dataset_Specific_Unlabelled.h5" -O unlabelled_data.h5

--2026-03-19 05:50:46--  https://portal.nersc.gov/cfs/m4392/G25/Dataset_Specific_Unlabelled.h5
Resolving portal.nersc.gov (portal.nersc.gov)... 128.55.206.106, 128.55.206.107, 128.55.206.108, ...
Connecting to portal.nersc.gov (portal.nersc.gov)|128.55.206.106|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 30000002048 (28G)
Saving to: ‘unlabelled_data.h5’

unlabelled_data.h5  100%[===================>]  27.94G  25.1MB/s    in 16m 58s 

2026-03-19 06:07:45 (28.1 MB/s) - ‘unlabelled_data.h5’ saved [30000002048/30000002048]



In [3]:
import h5py
import numpy as np

def inspect_h5_file(file_path):
    print(f"--- Inspecting {file_path} ---")
    try:
        with h5py.File(file_path, 'r') as f:
            for key in f.keys():
                data = f[key]
                print(f"Key: '{key}'")
                print(f"  Shape: {data.shape}")
                print(f"  Data Type: {data.dtype}")
                # Print the first item to see what the sparse data actually looks like
                print(f"  Sample (first element): \n{data[0]}\n")
    except Exception as e:
        print(f"Could not read as HDF5. Error: {e}")
        # Fallback in case it's an .npz file
        try:
            data = np.load(file_path, allow_pickle=True)
            for key in data.files:
                print(f"Key: '{key}' | Shape: {data[key].shape} | Type: {data[key].dtype}")
        except Exception as e2:
             print("File is neither standard HDF5 nor NPZ. Might be a ROOT file.")

# Inspect both
inspect_h5_file('unlabelled_data.h5')

--- Inspecting unlabelled_data.h5 ---
Key: 'jet'
  Shape: (60000, 125, 125, 8)
  Data Type: float32
  Sample (first element): 
[[[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 ...

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  ...
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0. 0.]]

 [[0. 0. 0. ... 0. 0. 0.]
  [0. 0. 0. ... 0. 0.

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import h5py
import numpy as np

class SparseJetDataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        # We open the file to get the length, but close it to avoid multiprocessing issues later
        with h5py.File(self.h5_path, 'r') as f:
            self.length = f['jet'].shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # 1. Lazy load ONLY the specific index we need
        with h5py.File(self.h5_path, 'r') as f:
            # Shape of single_event: (125, 125, 8)
            single_event = f['jet'][idx] 

        # 2. Sparsify the data! We only care about pixels where at least one channel is non-zero.
        # We sum across the 8 channels. If the sum is > 0, there is a hit in that pixel.
        hit_mask = np.sum(np.abs(single_event), axis=-1) > 0
        
        # Get the X, Y coordinates of the hits
        coords = np.argwhere(hit_mask)
        
        # Get the 8-channel features for those specific coordinates
        features = single_event[coords[:, 0], coords[:, 1]]
        
        # Convert to PyTorch tensors
        # coords shape will be: (Num_Hits, 2)
        # features shape will be: (Num_Hits, 8)
        return torch.tensor(coords, dtype=torch.int32), torch.tensor(features, dtype=torch.float32)

# --- Test the Dataloader ---
print("Initializing Dataset...")
dataset = SparseJetDataset('unlabelled_data.h5')

print(f"Total events: {len(dataset)}")

# Load the very first item
coords, features = dataset[0]

print("\n--- After Sparsification ---")
print(f"Instead of a massive 125x125x8 matrix (125,000 items),")
print(f"Event 0 only has {len(coords)} actual particle hits!")
print(f"Coordinates shape: {coords.shape}")
print(f"Features shape: {features.shape}")

Initializing Dataset...
Total events: 60000

--- After Sparsification ---
Instead of a massive 125x125x8 matrix (125,000 items),
Event 0 only has 1143 actual particle hits!
Coordinates shape: torch.Size([1143, 2])
Features shape: torch.Size([1143, 8])


In [2]:
!wget "https://portal.nersc.gov/cfs/m4392/G25/Dataset_Specific_labelled.h5" -O labelled_data.h5

--2026-03-22 05:28:48--  https://portal.nersc.gov/cfs/m4392/G25/Dataset_Specific_labelled.h5
Resolving portal.nersc.gov (portal.nersc.gov)... 128.55.206.112, 128.55.206.108, 128.55.206.110, ...
Connecting to portal.nersc.gov (portal.nersc.gov)|128.55.206.112|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5000042048 (4.7G)
Saving to: ‘labelled_data.h5’

labelled_data.h5    100%[===================>]   4.66G  52.5MB/s    in 6m 9s   

2026-03-22 05:34:57 (12.9 MB/s) - ‘labelled_data.h5’ saved [5000042048/5000042048]

